In [0]:
from pyspark.sql.functions import *

In [0]:
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<YOUR KEY>"
)


In [0]:
fact_sales_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/facts/fact_sales"
dim_customers_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/dimensions/dim_customers"

In [0]:
fs_df = spark.read.format("delta").load(fact_sales_path)

dim_customers_df = spark.read.format("delta").load(dim_customers_path)

In [0]:
fs_df.printSchema()

dim_customers_df.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- date_key: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- sale_key: long (nullable = true)

root
 |-- date: date (nullable = true)
 |-- date_key: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- week: integer (nullable = true)
 |-- day: integer (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: strin

In [0]:
state_revenue_base = fs_df.alias("f").join(dim_customers_df.alias('c'), on="customer_key", how="left")

In [0]:
state_revenue = state_revenue_base.groupBy("customer_state").agg(
    countDistinct("order_id").alias("total_orders"),
    sum("total_installments").alias("total_revenue")
)

In [0]:
display(state_revenue)

customer_state,total_orders,total_customers,total_revenue
pi,493,493,1982
pr,4998,4998,17851
rj,12762,12762,46555
pb,532,532,2376
ro,247,247,980
ba,3358,3358,13411
ms,709,709,2444
mg,11544,11544,41837
go,2007,2007,7557
sc,3612,3612,12746


In [0]:
from common_scripts.dq_framework import *

In [0]:
run_dq_check(
    state_revenue,
    "state_revenue",
    "null_total_revenue",
    col("total_revenue").isNull(),
    severity="critical"
)

run_dq_check(
    state_revenue,
    "state_revenue",
    "null_total_orders",
    col("total_orders").isNull(),
    severity="critical"
)


[CRITICAL] state_revenue | null_total_revenue: 0
[CRITICAL] state_revenue | null_total_orders: 0


0

In [0]:
dq_df = dq_df_from_dq_results(spark, dq_results)

dq_path = f"abfss://audit@{storage_account}.dfs.core.windows.net/gold_dq_logs/kpi_state_revenue"

dq_df.write.format('delta').mode('append').save(dq_path)

In [0]:
state_revenue_path =  f"abfss://gold@{storage_account}.dfs.core.windows.net/kpi/kpi_state_revenue"


state_revenue.write.format('delta').mode('overwrite').save(state_revenue_path)

In [0]:
sr = spark.read.format('delta').load(state_revenue_path)

display(sr)

customer_state,total_orders,total_revenue
pi,493,1982
pr,4998,17851
rj,12762,46555
ro,247,980
pb,532,2376
ba,3358,13411
ms,709,2444
mg,11544,41837
go,2007,7557
sc,3612,12746
